Generating an index from a csv with Fv sequences

In [ ]:
from abslang.build_search_index import build_index_from_csv

In [ ]:
#Create an output directory for the index and embeddings to be saved into

build_index_from_csv(
    csv_path="data/your_sequences.csv",                    
    id_column="PDB_file",                                  # Column name with sequence IDs
    seq_column="Sequence",                                 # Column name with sequences, format 'HC' + '|' + 'LC' for paired sequences. (Note: Only include the Fv sequence)
    out_dir="output/your_index_name",                      # output directory
    index_type="flat",                                     # 'flat', 'pq', or 'ivfpq'
    tm_checkpoint_path="models/paired_checkpoint_cpu.ckpt", # Path to transformer model checkpoint
    tm_config_path="models/params.json"                     # Path to transformer model config
)

Searching through a sequence database

In [ ]:
from abslang.search_antibody_index import FaissSearcher 

In [ ]:
searcher = FaissSearcher(
    faiss_index_file="paired_numbered_nonredundant_embeddings_QT4bit.index",
    csv_file="human_oas_paired_non_redundant_by_study.csv", 
    device="cpu",
    mode="paired",
    tm_checkpoint_path="models/paired_checkpoint_cpu.ckpt",  # checkpoint for the transformer model
    tm_config_path="models/params.json"                      # configuration for the transformer model
)


In [ ]:
query_sequence = "QVQLVESGGGVVQPGGSLRLSCAASGFTFSSYGMHWVRQAPGKGLEWVAFIRYDGSNKYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAKDSKLCGGDCYPSGRGYFDYWGQGTLVTVSS|QSALTQPRSVSGSPGQSVTISCTGTSSDVGGYNYVSWYQQHPGKAPKLMIYDVSKRPSGVPDRFSGSKSGNTASLTISGLQAEDEADYYCCSYAGSYTYVFGTGTKVTVL"
#query in the format 'HC' + '|' + 'LC'. (Note: Only include the Fv sequence)
results=searcher.search(query_sequence, k=10) #number of results to return
print(results)
import pandas as pd
results_table = pd.DataFrame(results)



In [ ]:
#Predict average CDR RMSD betweeen two sequences
from abslang.large_rmsd_inference import infer_rmsd, predict_cdr_rmsd, predict_cdr_rmsd_batch

In [ ]:
seq1 = 'QVKLQQSGAELVRSGTSVKLSCTASGFNIKDSYMHWLRQGPEQGLEWIGWIDPENGDTEYAPKFQGKATFTTDTSSNTAYLQLSSLTSEDTAVYYCNEGTPTGPYYFDYWGQGTTVTVSS|NVLTQSPAIMSASPGEKVTITCSASSSVSYMHWFQQKPGTSPKLWIYSTSNLASGVPARFSGSGSGTSYSLTISRMEAEDAATYYCQQRSSYPLTFGAGTKLELK'
seq2 = "QVQLQQWGAGLLKPSETLSLTCAVYGGSFSGSYWSWIRQPPGKGLEWIGEVNHSGSTNYNPSLKSRVTISVDTSKNHFSLKLSSVTAADTAVYYCSSVHLRFLEWIDDWGQGTLVTVSS|DIQMTQSPSSLSASVGDRVTITCRASKGIRNDLGWYQQKPGTAPKRLIYAASNLQSGVPSRFSGSGSGTEFTLTISSLQPEDFATYYCLQHNSYPQTFGQGTKVEIK"
predict_cdr_rmsd(seq1, seq2, device='cpu',
                tm_checkpoint_path="models/paired_checkpoint_cpu.ckpt",
                tm_config_path="models/params.json" )